In [8]:
!pip install youtube-transcript-api


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
from dotenv import load_dotenv
import os

In [3]:
load_dotenv()

True

In [4]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter, TextSplitter
from langchain_community.vectorstores import Chroma
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_huggingface.embeddings import HuggingFaceEndpointEmbeddings
from langchain_core.prompts import PromptTemplate

**LLM ENDPOINT**

In [5]:
llm = HuggingFaceEndpoint(repo_id=os.getenv('hf_model'))
hf_inf_model = ChatHuggingFace(llm=llm)

In [6]:
os.getenv('hf_emb')

'google/embeddinggemma-300m'

**EMBEDDING ENDPOINT**

In [6]:
hf_emb = HuggingFaceEndpointEmbeddings(repo_id=os.getenv('hf_emb'))

In [7]:
hf_emb.embed_query('ily')

[-0.21213452517986298,
 -0.008020435459911823,
 0.019333329051733017,
 -0.01114354096353054,
 0.016816115006804466,
 0.01709694415330887,
 -0.041050612926483154,
 0.03039906546473503,
 0.038961973041296005,
 -0.04673909768462181,
 -0.014967287890613079,
 -0.04456048831343651,
 0.04379810392856598,
 -0.0144955487921834,
 0.09401306509971619,
 0.013894880190491676,
 0.018724270164966583,
 -0.03216608613729477,
 -0.07122223824262619,
 -0.004377839621156454,
 0.012912933714687824,
 -0.015064035542309284,
 -0.011184394359588623,
 -0.018372301012277603,
 -0.0014006610726937652,
 0.02646259218454361,
 0.025507763028144836,
 0.007077949121594429,
 -0.009125813841819763,
 -0.040779538452625275,
 0.03199100121855736,
 0.01577100157737732,
 0.026917537674307823,
 0.010337743908166885,
 -0.009550299495458603,
 0.06451431661844254,
 0.030345836654305458,
 -0.08022163063287735,
 0.051793087273836136,
 -0.007865555584430695,
 -0.0740189254283905,
 0.04730917885899544,
 -0.002594812074676156,
 -0.0181

**VECTOR STORE**

In [10]:
vectorstore = Chroma(
    persist_directory='youtube_transcript',
    embedding_function=hf_emb,
    collection_name='youtube'
)

**DATALOADER**

In [17]:
video_id = 'S3qOEWeEYgM'
try: 
    youtube_transcript_obj = YouTubeTranscriptApi()
    fetched_content = youtube_transcript_obj.fetch(video_id=video_id, languages=['en'])
except TranscriptsDisabled:
    print('no transcript')

In [25]:
fetched_content_transcript = " ".join(snippet.text for snippet in fetched_content.snippets)

**TEXT SPLITTER**

In [33]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)

In [42]:
text_chunks = splitter.split_text(fetched_content_transcript)

In [45]:
text_chunk_docs = splitter.create_documents(text_chunks) # converting the text split into docs, easy to insert in vector db

**VECTOR STORE DATA INSERTION / INDEXING**

In [46]:
vectorstore.add_documents(text_chunk_docs)

['fa53cb19-4a35-4984-a291-13df1edee9e4',
 '41deba4f-4670-4f03-90c5-dc0016019da4',
 'a53bdac4-1c6f-4216-9cca-c1076c741e2f',
 '3edd840b-6d73-4860-b7c9-e21910170a50',
 'f07989c6-c584-4ad7-ab4e-558782fda9b5',
 '8bf5d3b7-9478-439d-9833-a6af2fb7f122',
 '01a5ac0d-6ac4-4d84-b9da-b1056fe600f9',
 'a06a001c-c6fd-4009-be2c-75dd4c94bd09',
 '373182ab-1ee9-4a04-b5dd-123052ca90e7',
 'fe5614e4-4576-4682-b995-e3ac3b5d6d45',
 'e9291300-1c48-4010-a183-027edbefe650',
 '43bbe6b1-b3b3-4bed-af5d-306c85785ed2',
 'facfd825-b0e8-45f2-b50c-13974f8ef509',
 '47f162e4-d063-4474-94f2-95101e5cd484',
 'c24d286d-662b-4ebc-ba41-4335ea861189']

**RETRIEVER**

In [50]:
retriever = vectorstore.as_retriever(search_kwargs={'k':2})

In [51]:
retriever.invoke('morality')

[Document(metadata={}, page_content="and hopes the head of the guyu clan had a middle-aged appearance his sideburns were graying and he was clothed in ceremonial white robes kneeling on the brownish yellow floor his body was straight with his hands held together eyes tightly shut as he prayed sincerely he was facing a tall black case there were three layers on the case all housing Memorial tablets of ancestors on both sides of the tablets was copper incense the smoke Rising behind him were over 10 people kneeling in a similar fashion as him they wore loose white ceremonial garments and were all the Clan's Elders important members and those who had much Authority after finishing prayers the guu clan had bent his waist with his two hands pressing against the floor and cout out as the forehead knocked against the brownish yellow floor light thuds could be heard behind him the elders and important clan members solemnly and quietly followed suit with this the hall was filled with light thud

**AUGMENTATION OF RETRIEVED DOCS TO QUERY - PROMPT**

In [52]:
prompt_template = PromptTemplate(
    template="""
    you are an helpful assistant,
    answer only from the context.
    if the context is insuffiecient, just say you dont know.
    context:{context}
    question:{query}
    """,
    input_variables=['context','query']
)

In [65]:
question = 'bai ning bing'
retrieved_docs = retriever.invoke(question)

In [66]:
retrieved_text = "\n\n ".join(doc.page_content for doc in retrieved_docs)
retrieved_text

"it was already late in the night a slight breeze blowing with the light rain yet Ching Mountain was was not covered in darkness from the side down to the foot of the mountain dozens of tiny lights Shone like a bright band these lights Shone from tall buildings even though it could not be said to match up to 10,000 lights yet it was still a few thousand in number situated on the mountain was guu one Village giving the vast Lonely Mountain a rich Touch of human civilization in the middle of the guyu village was a magnificent Pavilion a grand ceremony was being held at this moment and the lights were even brighter than ever radiating with Glory ancestors please bless us we pray that this ceremony will bring many young men of outstanding talent and intelligence bringing their families new blood and hopes the head of the guyu clan had a middle-aged appearance his sideburns were graying and he was clothed in ceremonial white robes kneeling on the brownish yellow floor his body was straight\

In [67]:
prompt = prompt_template.invoke({'context':retrieved_text,'query':question})

In [68]:
response = hf_inf_model.invoke(prompt)

In [69]:
response

AIMessage(content='I don\'t know. The context provided does not contain information related to "bai ning bing."', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 462, 'total_tokens': 483}, 'model_name': 'Qwen/Qwen2.5-7B-Instruct', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019ffa07-c4b0-7f83-9435-df19121fd928-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 462, 'output_tokens': 21, 'total_tokens': 483})

**BUILDING CHAIN**

In [76]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda, RunnableSequence
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser

In [77]:
def txt_return_func(docs):
    content = "\n\n ".join(doc.page_content for doc in retrieved_docs)
    return content

In [78]:
parallel_chain = RunnableParallel({
    'context':retriever | RunnableLambda(txt_return_func),
    'query':RunnablePassthrough()
}
)

In [85]:
parser = StrOutputParser()

In [90]:
par_resp = parallel_chain.invoke('who is fang yuan')

In [93]:
main_chain = parallel_chain | prompt_template | hf_inf_model | parser

In [99]:
user_question = 'can you summarize entire video context'

In [100]:
main_chain.invoke(user_question)

'The context describes a nighttime scene on Ching Mountain, where the mountain is illuminated by numerous small lights from tall buildings, giving it a touch of human civilization. In the middle of Guyu Village, a grand ceremony is being held inside a magnificent pavilion. The hall is filled with light thuds as the clan members, including the head of the clan and his elders, kneel and bow their foreheads against the floor, praying for the blessings of their ancestors and the arrival of promising young talents. The head of the clan, a middle-aged man with graying sideburns, is kneeling in ceremonial white robes, his hands held together and eyes tightly shut. The important members are also kneeling in similar attire, and the scene is filled with the rising incense smoke from the tall black case holding memorial tablets of ancestors.'

**WHATEVER STUFF**

In [102]:
from langchain_community.tools import DuckDuckGoSearchRun

In [106]:
ddsr = DuckDuckGoSearchRun()

In [107]:
ddsr.invoke('current nifty50 price')

"NIFTY 50 Share Price Chart - View today’s NIFTY 50 Stock Price Chart for BSE and NSE at Groww Terminal. See Nifty 50 PE today, with live chart. Compare NSE Nifty 50 by performance, price, EPS, dividend yield.Nifty 50 Price to Earnings for index companies, and analysis of constituents. The current GIFT Nifty live price is shown in the widget at the top of this page, updated in real time during both trading sessions. You can also track it on NSE IX's official website (nseix.com) or major financial platforms. Explore the latest information on LTIMindtree, including: Last traded price 4835.5, Market capitalization: 143546.92, Volume: 320933, Price-to-earnings ratio 27.44, Earnings per share 176.33. Prev Day Close. 23853.9. NIFTY Company Information.Max Healthcare Institute Ltd. 1023.50."

In [108]:
from langchain_community.tools import ShellTool

In [109]:
shelltool = ShellTool()

In [112]:
shelltool.invoke('whoami')

Executing command:
 whoami


'desktop-pdegih5\\rudeu\r\n'

In [123]:
from langchain_core.tools import BaseTool,tool

In [115]:
@tool
def tenX(a:int) -> int:
    """multiplies a num with 10"""
    return a*10    

In [117]:
tenX.invoke({'a':3})

30

In [118]:
tenX.name

'tenX'

In [120]:
tenX.description

'multiplies a num with 10'

In [121]:
tenX.args

{'a': {'title': 'A', 'type': 'integer'}}

In [122]:
tenX.args_schema.model_json_schema()

{'description': 'multiplies a num with 10',
 'properties': {'a': {'title': 'A', 'type': 'integer'}},
 'required': ['a'],
 'title': 'tenX',
 'type': 'object'}